# Modules in Python

Any python script written in file with .py extension can be called as a module. Modules are used to break down large programs into small manageable and organized files.

The main benifit of the modules is that it allows us to reuse the code. We can define our most used functions in a module and import that module, instead of copying their definitions into different programs. A module can contain functions, classes and variables. A module can also include runnable code.

The module name is simply the name of the file without the .py extension. There is one built-in variable in python called `__name__` which evaluates to the name of the current module, for example if you imported one module name `fibo` so when the execution of that module is done, `__name__` will be set to `fibo`. However, if a module is being run directly (as the main program), then `__name__` will be set to `__main__`. This allows us to check whether a module is being run directly or being imported, and execute code accordingly.

Each module has its own private symbol table, which is used as the global symbol table by all functions defined in the module. Thus, we can use the module name to access the functions and variables defined in the module.

In many codebases you will find the following code at the end of the file: 

```python
if __name__ == "__main__":
    #code to be executed when the module is run directly
```

This condition is used because the devloper wants to execute some code only when the module is run directly, and not when it is imported. This is a common practice in python and is used to test the module or to provide a command-line interface for the module.

find mode details about this [from here](https://youtu.be/sugvnHA7ElY)

In [4]:
import test
print(test.__name__)

print("Hello")
print(__name__)

test
Hello
__main__


## Importing Modules

### import _module_name_

__This does not enter the names of the functions defined in fibo directly in the current symbol table; it only enters the module name fibo there. Using the module name you can access the functions:__

- Meaning that any change made using fibo.some_name syntax will affect the scope of fibo module.

- If you intend to use a function often you can assign it to a local name:

```python
fib = fibo.fib
fib(500)
```

### from _module_ import _name1, name2_

- Imports names from a module directly into the **importing module’s symbol table.**

- **This does not introduce the module name from which the imports are taken in the local symbol table (so in the example, fibo is not defined).**

- Meaning that any change made to the imported names will not affect the scope of fibo module.

```python
from math import sqrt, ceil
sqrt(2)
ceil(2.5)
```

### from _module_ import _*_

- This imports all names except those beginning with an underscore (_)

- **Do not use this facility since it introduces an unknown set of names into the interpreter.**

- In general the practice of importing * from a module or package is frowned upon, since it often causes poorly readable code.

```python
#Avoid this kind of import as much as possible, because it can lead to confusion and bugs in the code.
from math import *
```

### import _module_ as _name_

- This is effectively importing the module in the same way that import fibo will do, with the only difference of it being available as _name_ that you used after 'as'.

- It can also be used when utilising from with similar effects

## Executing modules as scripts

- the code in the module will be executed, just as if you imported it

- but with the **\_\_name\_\_ set to "\_\_main\_\_"**. That means that by adding this code at the end of your module:

```python
if __name__ == "__main__":
    import sys
    fib(int(sys.argv[1]))
```

- Now you can do like:
```bash
python fibo.py 50
```

## Module Search Path

When you import some module named as `foo` the searching sequance that interpreter will follow is:

1) **Built-in Modules**: Checks if it's part of Python itself.<br>

These module names are listed in `sys.builtin_module_names`, it then searches for a file named `foo.py` in a list of directories given by the variable `sys.path`, which follows this sequence:

2) **Current Directory**: Checks the folder where your script is running.

3) **PYTHONPATH**: Checks any custom folders you added to your environment variables.

4) **Installation Defaults**: Checks the standard library folders and third-party packages installed in your Python environment.

**Note** : On file systems which support symlinks, the directory containing the input script is calculated after the symlink is followed. In other words the directory containing the symlink is not added to the module search path.

After initialization, Python programs can modify `sys.path`, The directory containing the current script will placed at the beginning of the search path, ahead of the standard library path. This means that scripts in that directory will be loaded instead of modules of the same name in the library directory. This is an error unless the replacement is intended.

If it found the module, it will be loaded and initialized. If it cannot find the module, it raises a `ModuleNotFoundError` exception.


## Circular Import

Circular imports occur when two or more modules attempt to import each other. This can lead to a situation where the modules are trying to access each other's contents before they have been fully initialized, resulting in an ImportError.

for example, if module A imports module B, and module B imports module A.

**The Deadlock**: Python starts reading a.py, sees import b, pauses a, goes to b, sees import a... and gets stuck (or fails) because a isn't finished loading yet.

The solution of circular import is possible, but avoid it if you can.

### Core Logic of modules

Before we move to the solution just read this bit that I've taken from [this](https://groups.google.com/g/comp.lang.python/c/HYChxtsrhnw) Google Groups discussion:

Imports are pretty straightforward really. Just remember the following:

`import` and `from xxx import yyy` are executable statements. They execute
when the running program reaches that line.

If a module is not in `sys.modules`, then an import creates the new module
entry in `sys.modules` and then executes the code in the module. It does not
return control to the calling module until the execution has completed.

If a module does exist in `sys.modules` then an import simply returns that
module whether or not it has completed executing. That is the reason why
cyclic imports may return modules which appear to be partly empty.

Finally, the executing script runs in a module named `__main__`, importing
the script under its own name will create a new module unrelated to
`__main__`.

Take that lot together and you shouldn't get any surprises when importing
modules.

### Solutions of circular imports

1) **Importing inside a function**: You can import the module inside a function, so that it is only imported when the function is called. This can help to avoid circular imports because the import will not be executed until the function is called.

2) **Using importlib**: You can use the importlib module to import a module at runtime. This can help to avoid circular imports because the import will not be executed until the module is needed.

Check out this two files [a.py](./modules/a.py) and [b.py](./modules/b.py) for example of circular import and its solution.

I'm running the `a.py` file and we will then talk about the output and how it is working.

In [ ]:
import os

os.chdir("./modules")
os.system("python a.py")

out side a's display method
out side b's display method
out side a's display method
in side b's display method
in side a's display method
in side b's display method


0

**1. Start `a.py` (Running as `__main__`)**
* **Action:** You run the script from the command line.
* **Context:** Python names this specific execution instance `__main__`. It does *not* know it is named "a" yet.
* **Output:** Prints `"Outside a's display method"`.
* **Status:** It defines the functions `show` and `display`, then calls `show()` at the bottom of the file.

**2. Inside `a.py`'s `show()`**
* **Action:** The line `import b` is executed.
* **Logic:** Python checks `sys.modules`. It does not find `b`.
* **Result:** Python pauses the execution of `__main__` to go find and load `b.py`.

**3. Start `b.py` (Initializing module `b`)**
* **Action:** Python executes `b.py` from top to bottom to initialize it.
* **Context:** A new entry `b` is added to `sys.modules`.
* **Output:** Prints `"Outside b's display method"`.
* **Status:** It defines its functions, then calls its own `show()` at the bottom.

**4. Inside `b.py`'s `show()`**
* **Action:** The line `import a` is executed.
* **Logic:** Python checks `sys.modules` for a module named `a`.
* **Crucial Detail:** It does **not** find `a`. It only finds `__main__` and `b`. Python does not realize that `__main__` is actually the file `a.py`.
* **Result:** Python finds `a.py` on the disk and begins loading it *again* as a completely new module named `a`.

**5. Start `a.py` (Running as module `a`)**
* **Action:** `a.py` is executed a second time.
* **Context:** This instance is named `a`, not `__main__`.
* **Output:** Prints `"Outside a's display method"` (for the second time).
* **Status:** It defines functions and calls `show()` again.

**6. Inside `a.py`'s `show()` (The nested run)**
* **Action:** The line `import b` is executed.
* **Logic:** Python checks `sys.modules`. It finds `b` (created in Step 3).
* **Result:** The import succeeds immediately.
* **Action:** It calls `b.display()`.
* **Output:** Prints `"Inside b's display method"`.
* **Status:** This nested execution of `a.py` finishes.

**7. Unwinding the Stack**
* **Action:** Control returns to `b.py` (Step 4) now that `import a` is complete.
* **Context:** We are back in `b.show()`.
* **Action:** It calls `a.display()`.
* **Output:** Prints `"Inside a's display method"`.
* **Status:** `b.py` finishes its initialization. Control returns to the original `__main__`.

**8. Back in `a.py` (`__main__`)**
* **Action:** Control returns to the original script (Step 2) now that `import b` is complete.
* **Context:** We are back in `__main__`'s `show()`.
* **Action:** It calls `b.display()`.
* **Output:** Prints `"Inside b's display method"`.
* **Result:** The program finishes.

----

# Packages

- Collection of modules (.py files) is called a package.
- For a directory to be considered as a package it must have `__init__.py` file.
- Most of the time you can leave `__init__.py` empty.

## Importing from packages

Importing from packages is almost same as modules. although there are some differences.

**from _package_ import \***

- If a package’s `__init__.py` code defines a list named `__all__`, it is taken to be the list of module names that should be imported when from package import * is encountered.
- If `__all__` is not defined in `__init__.py` then those modules which are already imported by some other module in current python session, are imported.
- In nutshell when using `from package import *` the module to be imported either should be in `__all__` list or it should have been already imported some place else.

### Library

You often will hear the term library in python. you can think of library as a big package that contains other packages and modules. There is no any formal definition of library in python.

### The dir() function

The `dir()` function is a built-in function in Python that returns a list of the attributes and methods of an object. When you call `dir()` on a module, it will return a list of all the names defined in that module, including functions, classes, and variables. This can be useful for exploring the contents of a module and understanding what it provides.

In [11]:
import test
dir(test)

['__builtins__',
 '__cached__',
 '__doc__',
 '__file__',
 '__loader__',
 '__name__',
 '__package__',
 '__path__',
 '__spec__']

In [12]:
import itertools
dir(itertools)

['__doc__',
 '__loader__',
 '__name__',
 '__package__',
 '__spec__',
 '_grouper',
 '_tee',
 '_tee_dataobject',
 'accumulate',
 'batched',
 'chain',
 'combinations',
 'combinations_with_replacement',
 'compress',
 'count',
 'cycle',
 'dropwhile',
 'filterfalse',
 'groupby',
 'islice',
 'pairwise',
 'permutations',
 'product',
 'repeat',
 'starmap',
 'takewhile',
 'tee',
 'zip_longest']

### \_\_pycache\_\_ directory

To speed up loading modules, Python caches the compiled version of each module in the __pycache__ directory under the name module.version.pyc, where the version encodes the format of the compiled file; it generally contains the Python version number. For example, in CPython release 3.3 the compiled version of spam.py would be cached as __pycache__/spam.cpython-33.pyc. This naming convention allows compiled modules from different releases and different versions of Python to coexist.

Python checks the modification date of the source against the compiled version to see if it’s out of date and needs to be recompiled. This is a completely automatic process. Also, the compiled modules are platform-independent, so the same library can be shared among systems with different architectures.

Python does not check the cache in two circumstances. First, it always recompiles and does not store the result for the module that’s loaded directly from the command line. Second, it does not check the cache if there is no source module. To support a non-source (compiled only) distribution, the compiled module must be in the source directory, and there must not be a source module